# 08 - CTGAN on the leader's `smote_10k` file: tune (Optuna) + final training + validity check

Leader's `final_datasets/generators/CTGAN/unsw_nb15_attacks_smote_10k.csv` -> **one** CTGAN.

1. **Tune** with Optuna on a small class-balanced sample (cap per class), search space narrowed around the CTGAN defaults
   (`batch_size` 500/1000, `discriminator_steps` 1/3).
2. **Train the final CTGAN** on the **full** file, save the `.pt` and the synthetic CSV.
3. **Validity check**: share of invalid rows (negative values / non-integer values in integer columns / NaN) of the output.

`fix_pipeline.py` must be in `src/` (same as notebook 07). `pip install optuna ctgan` first.
Tuning can run on CPU (`TUNE_GPU = False`); the final training and sampling use the GPU when there is one.
Keep `QUICK = True` for a first test (a few minutes).

In [1]:
import os, sys, time, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np, pandas as pd, torch, optuna
from sklearn.model_selection import train_test_split
from src import fix_pipeline as f

print("CUDA available:", torch.cuda.is_available())

# ---- CONFIG ----
QUICK = True            # True: tiny test run. False: real run
SEED = 42
TUNE_GPU = False        # tuning trials on CPU (small data); set True to test on GPU
TRAIN_CSV = Path(os.environ.get("TRAIN_CSV", ROOT / "dataset" / "UNSW_NB15" / "final" / "unsw_nb15_attacks_smote_10k.csv"))
OUT_DIR = Path(os.environ.get("OUT_DIR", ROOT / "outputs")); OUT_DIR.mkdir(parents=True, exist_ok=True)

if QUICK:
    N_TRIALS, EPOCH_MAX, PER_CLASS, FINAL_EPOCHS, TAG = 2, 20, 300, 5, "smote10k_quick"
else:
    N_TRIALS, EPOCH_MAX, PER_CLASS, FINAL_EPOCHS, TAG = 8, 150, 1500, 50, "smote10k"
print(dict(N_TRIALS=N_TRIALS, EPOCH_MAX=EPOCH_MAX, PER_CLASS=PER_CLASS, FINAL_EPOCHS=FINAL_EPOCHS, TAG=TAG, TUNE_GPU=TUNE_GPU))

def invalid_rate(df, real, label=f.LABEL):
    """Share of rows that are invalid: negative where the real column is non-negative,
    non-integer where the real column is integer, NaN or inf."""
    feats = [c for c in real.columns if c != label and c in df.columns]
    bad = pd.Series(False, index=df.index)
    for c in feats:
        r = real[c].replace([np.inf, -np.inf], np.nan).dropna()
        x = df[c]
        if r.min() >= 0:                 bad |= x < 0
        if np.all(np.isclose(r % 1, 0)): bad |= ~np.isclose(x % 1, 0)
        bad |= x.isna() | np.isinf(x)
    return round(float(bad.mean()), 4)

c:\Users\Admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CUDA available: False
{'N_TRIALS': 2, 'EPOCH_MAX': 20, 'PER_CLASS': 300, 'FINAL_EPOCHS': 5, 'TAG': 'smote10k_quick', 'TUNE_GPU': False}


## 1. Tuning data: class-balanced sample, split into train / hold-out (the generator never sees the hold-out)

Note: the file contains SMOTE rows, so a hold-out cut from it is optimistic (neighbours of the training rows). It is good enough to RANK settings, not to report absolute quality. The leader's `holdout_set` is not used here, so it stays untouched.

In [2]:
full = pd.read_csv(TRAIN_CSV)
print("full file:", full.shape)
display(full[f.LABEL].value_counts())
sample = full.sample(frac=1, random_state=SEED).groupby(f.LABEL).head(PER_CLASS).reset_index(drop=True)
tune_train, tune_test = train_test_split(sample, test_size=0.3, stratify=sample[f.LABEL], random_state=SEED)
print("tuning train:", tune_train.shape, "| hold-out:", tune_test.shape)

full file: (123977, 49)


Attack
Exploits          32390
Fuzzers           21587
Reconnaissance    10000
DoS               10000
Analysis          10000
Generic           10000
Backdoor          10000
Shellcode         10000
Worms             10000
Name: count, dtype: int64

tuning train: (1890, 49) | hold-out: (810, 49)


## 2. Optuna (TPE) with the compound score = mean(label consistency macro, TSTR macro-F1)

In [3]:
DIMS = {"256x256": (256, 256), "256x256x256": (256, 256, 256), "512x512": (512, 512)}

def params_from(trial):
    return dict(
        epochs=trial.suggest_int("epochs", 50, EPOCH_MAX, step=50) if EPOCH_MAX >= 100 else trial.suggest_int("epochs", 10, EPOCH_MAX, step=10),
        batch_size=trial.suggest_categorical("batch_size", [500, 1000]),
        pac=trial.suggest_categorical("pac", [1, 5, 10]),
        embedding_dim=trial.suggest_categorical("embedding_dim", [64, 128, 256]),
        generator_dim=DIMS[trial.suggest_categorical("generator_dim", list(DIMS))],
        discriminator_dim=DIMS[trial.suggest_categorical("discriminator_dim", list(DIMS))],
        generator_lr=trial.suggest_float("generator_lr", 1e-5, 1e-3, log=True),
        discriminator_lr=trial.suggest_float("discriminator_lr", 1e-5, 1e-3, log=True),
        discriminator_steps=trial.suggest_categorical("discriminator_steps", [1, 3]),
        log_frequency=trial.suggest_categorical("log_frequency", [True, False]),
    )

def objective(trial):
    pr = params_from(trial)
    t0 = time.time()
    _, _, syn = f.train_and_sample(tune_train, seed=SEED, verbose=False, enable_gpu=TUNE_GPU, **pr)
    res, _ = f.evaluate(tune_train, tune_test, syn, seed=SEED)
    for k in ("label_consistency_macro", "tstr_macro_f1"): trial.set_user_attr(k, res[k])
    trial.set_user_attr("minutes", round((time.time() - t0) / 60, 1))
    print(f"trial {trial.number}: score={res['score']} ({round((time.time()-t0)/60,1)} min)")
    return res["score"]

optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False)

trials = study.trials_dataframe()
display(trials.sort_values("value", ascending=False))
trials.to_csv(OUT_DIR / f"optuna_trials_{TAG}.csv", index=False)
print("BEST score:", round(study.best_value, 4))
print("BEST params:", study.best_params)

  rows missing (generator ignored the condition): {'Fuzzers': 7, 'Exploits': 24, 'DoS': 47, 'Analysis': 33, 'Generic': 2, 'Worms': 27, 'Reconnaissance': 66, 'Backdoor': 3, 'Shellcode': 42}
trial 0: score=0.1243 (1.2 min)
  rows missing (generator ignored the condition): {'Fuzzers': 10, 'Exploits': 23, 'DoS': 35, 'Analysis': 31, 'Generic': 32, 'Worms': 34, 'Reconnaissance': 24, 'Backdoor': 30}
trial 1: score=0.0922 (0.6 min)


,number,value,datetime_start,datetime_complete,duration,params_batch_size,params_discriminator_dim,params_discriminator_lr,params_discriminator_steps,params_embedding_dim,params_epochs,params_generator_dim,params_generator_lr,params_log_frequency,params_pac,user_attrs_label_consistency_macro,user_attrs_minutes,user_attrs_tstr_macro_f1,state
0,0,0.1243,2026-11-02 09:15:44.530329,2026-11-02 09:16:57.257560,0 days 00:01:12.727231,500,256x256,0.000041,1,128,10,512x512,0.000023,False,1,0.0994,1.2,0.1491,COMPLETE
1,1,0.0922,2026-11-02 09:16:57.257560,2026-11-02 09:17:33.879513,0 days 00:00:36.621953,1000,256x256x256,0.000016,1,128,10,256x256,0.000041,False,5,0.1130,0.6,0.0714,COMPLETE


BEST score: 0.1243
BEST params: {'epochs': 10, 'batch_size': 500, 'pac': 1, 'embedding_dim': 128, 'generator_dim': '512x512', 'discriminator_dim': '256x256', 'generator_lr': 2.3270677083837777e-05, 'discriminator_lr': 4.059611610484306e-05, 'discriminator_steps': 1, 'log_frequency': False}


## 3. Final CTGAN on the FULL file (GPU if available)

`FINAL_EPOCHS` replaces the tuned `epochs` (one epoch on the full file has many more steps than on the tuning sample). Check the time of the first run before raising it.

In [4]:
best = dict(study.best_params)
tuned_epochs = best["epochs"]
best["generator_dim"] = DIMS[best.pop("generator_dim")]
best["discriminator_dim"] = DIMS[best.pop("discriminator_dim")]
best["epochs"] = FINAL_EPOCHS
print("final parameters:", best, "| epochs found while tuning:", tuned_epochs)
json.dump({**{k: (list(v) if isinstance(v, tuple) else v) for k, v in best.items()}, "tuned_epochs": tuned_epochs},
          open(OUT_DIR / f"best_params_{TAG}.json", "w"), indent=1)

real = full if not QUICK else full.sample(frac=1, random_state=SEED).groupby(f.LABEL).head(PER_CLASS).reset_index(drop=True)
t0 = time.time()
model, p, syn = f.train_and_sample(real, seed=SEED, verbose=True, enable_gpu=torch.cuda.is_available(), **best)
minutes = round((time.time() - t0) / 60, 1)
model.save(str(OUT_DIR / f"ctgan_unsw_nb15_smote_10k_{TAG}.pt"))
syn.to_csv(OUT_DIR / f"synthetic_ctgan_unsw_nb15_smote_10k_{TAG}.csv", index=False)
print("done in", minutes, "min | synthetic rows:", len(syn))

final parameters: {'epochs': 5, 'batch_size': 500, 'pac': 1, 'embedding_dim': 128, 'generator_lr': 2.3270677083837777e-05, 'discriminator_lr': 4.059611610484306e-05, 'discriminator_steps': 1, 'log_frequency': False, 'generator_dim': (512, 512), 'discriminator_dim': (256, 256)} | epochs found while tuning: 10


Gen. (+02.82) | Discrim. (-00.06): 100%|██████████| 5/5 [00:08<00:00,  1.61s/it]


  rows missing (generator ignored the condition): {'Analysis': 42, 'Exploits': 21, 'Worms': 26, 'Fuzzers': 45, 'Backdoor': 25, 'Reconnaissance': 27, 'Shellcode': 59, 'Generic': 53, 'DoS': 42}
done in 0.6 min | synthetic rows: 2360


## 4. Checks: validity (leader's requirement) and label consistency

In [5]:
ok, per_class = f.label_consistency(real, syn, seed=SEED)
num = syn.select_dtypes("number")
summary = pd.Series({
    "real_rows": len(real), "synthetic_rows": len(syn), "minutes": minutes,
    "invalid_row_rate": invalid_rate(syn, real),
    "negative_cells": int((num < 0).sum().sum()), "NaN_cells": int(syn.isna().sum().sum()),
    "cat_validity_min": float(f.cat_validity(real, syn, p).min()),
    "label_consistency_macro": round(float(per_class.mean()), 3),
}, name="smote_10k")
display(summary.to_frame())

comp = pd.DataFrame({"real_rows": real[f.LABEL].value_counts(), "synthetic_rows": syn[f.LABEL].value_counts(),
                     "label_consistency": per_class})
comp["missing_rows"] = comp["real_rows"] - comp["synthetic_rows"]
display(comp.round(3))

summary.to_frame().to_csv(OUT_DIR / f"summary_{TAG}.csv")
comp.round(3).to_csv(OUT_DIR / f"per_class_{TAG}.csv")

,smote_10k
real_rows,2700.0
synthetic_rows,2360.0
minutes,0.6
invalid_row_rate,0.0
negative_cells,0.0
NaN_cells,0.0
cat_validity_min,1.0
label_consistency_macro,0.1


,real_rows,synthetic_rows,label_consistency,missing_rows
Attack,,,,
Analysis,300,258,0.000,42
Backdoor,300,275,0.124,25
DoS,300,258,0.593,42
Exploits,300,279,0.036,21
Fuzzers,300,255,0.035,45
Generic,300,247,0.101,53
Reconnaissance,300,273,0.004,27
Shellcode,300,241,0.004,59
Worms,300,274,0.007,26


## How to read
* `invalid_row_rate` should be ~0 (clamping + integer rounding are applied inside `fix_pipeline.inverse`). Report this number to the leader.
* `label_consistency_macro`: chance level with 9 classes is about 0.11. The earlier 10-epoch CTGANs scored 0.12-0.13 (close to chance); compare against that.
* `missing_rows` > 0 means the generator ignored the class condition for that class.
* The hold-out used for tuning is cut from the (SMOTE) training file, so scores are only for ranking settings.
* `QUICK = True` results only test that the notebook runs.